# Week 3 Assignment: Customer Intelligence System

**Objective:** Build an end-to-end intelligence system using the provided country dataset and data dictionary. The workflow covers EDA, customer/country segmentation with K-Means and DBSCAN, and predictive classification using Logistic Regression, Random Forest, and XGBoost when available.

Because the supplied dataset contains country-level socio-economic indicators rather than direct customer records, each country is treated as a customer/market entity for segmentation and prioritization.

In [1]:
from pathlib import Path
import warnings

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, adjusted_rand_score, classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')

In [2]:
ROOT = Path.cwd()
DATA_PATH = ROOT / 'Country-data.csv'
DICTIONARY_PATH = ROOT / 'data-dictionary.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'customer_intelligence'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

features = [
    'child_mort', 'exports', 'health', 'imports', 'income', 'inflation',
    'life_expec', 'total_fer', 'gdpp'
]

df = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)

print('Dataset shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
display(dictionary)
display(df.head())

Dataset shape: (167, 10)
Missing values: 0


,Column Name,Description
0,country,Name of the country
1,child_mort,Death of children under 5 years of age per 100...
2,exports,Exports of goods and services per capita. Give...
3,health,Total health spending per capita. Given as %ag...
4,imports,Imports of goods and services per capita. Give...
5,Income,Net income per person
6,Inflation,The measurement of the annual growth rate of t...
7,life_expec,The average number of years a new born child w...
8,total_fer,The number of children that would be born to e...
9,gdpp,The GDP per capita. Calculated as the Total GD...


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200


## Exploratory Data Analysis

In [3]:
summary = df[features].describe().T
summary.to_csv(OUTPUT_DIR / 'eda_summary.csv')
display(summary)

plt.figure(figsize=(11, 8))
sns.heatmap(df[features].corr(), annot=True, cmap='vlag', center=0, fmt='.2f')
plt.title('Country Indicator Correlation Heatmap')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmap.png', dpi=180)
plt.show()

,count,mean,std,min,25%,50%,75%,max
child_mort,167.0,38.270060,40.328931,2.6000,8.250,19.30,62.10,208.00
exports,167.0,41.108976,27.412010,0.1090,23.800,35.00,51.35,200.00
health,167.0,6.815689,2.746837,1.8100,4.920,6.32,8.60,17.90
imports,167.0,46.890215,24.209589,0.0659,30.200,43.30,58.75,174.00
income,167.0,17144.688623,19278.067698,609.0000,3355.000,9960.00,22800.00,125000.00
inflation,167.0,7.781832,10.570704,-4.2100,1.810,5.39,10.75,104.00
life_expec,167.0,70.555689,8.893172,32.1000,65.300,73.10,76.80,82.80
total_fer,167.0,2.947964,1.513848,1.1500,1.795,2.41,3.88,7.49
gdpp,167.0,12964.155689,18328.704809,231.0000,1330.000,4660.00,14050.00,105000.00


## Feature Engineering and Scaling

Exports, imports, and health are percentages of GDP per capita, so actual per-capita values are also created for interpretation. Clustering is performed on standardized features.

In [4]:
df_model = df.copy()
df_model['exports_value'] = df_model['exports'] * df_model['gdpp'] / 100
df_model['imports_value'] = df_model['imports'] * df_model['gdpp'] / 100
df_model['health_value'] = df_model['health'] * df_model['gdpp'] / 100

scaler = StandardScaler()
x_scaled = scaler.fit_transform(df_model[features])

## K-Means Clustering

Different cluster counts are compared using inertia and silhouette score. A final value of **k=3** is used because it gives interpretable business groups: high priority, developing, and stable/developed.

In [5]:
k_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    labels = km.fit_predict(x_scaled)
    k_rows.append({'k': k, 'inertia': km.inertia_, 'silhouette_score': silhouette_score(x_scaled, labels)})

kmeans_results = pd.DataFrame(k_rows)
kmeans_results.to_csv(OUTPUT_DIR / 'kmeans_k_selection.csv', index=False)
display(kmeans_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=kmeans_results, x='k', y='inertia', marker='o', ax=axes[0])
axes[0].set_title('K-Means Elbow Curve')
sns.lineplot(data=kmeans_results, x='k', y='silhouette_score', marker='o', ax=axes[1])
axes[1].set_title('K-Means Silhouette Score')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kmeans_selection.png', dpi=180)
plt.show()

,k,inertia,silhouette_score
0,2,1050.214558,0.287357
1,3,831.424435,0.283296
2,4,700.391720,0.302108
3,5,620.163371,0.299259
4,6,550.877188,0.230149
5,7,495.535861,0.249890
6,8,457.586148,0.238811


In [6]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=30)
df_model['kmeans_cluster'] = kmeans.fit_predict(x_scaled)

profiles = df_model.groupby('kmeans_cluster')[features].mean()
risk_score = (
    profiles['child_mort'].rank(ascending=False)
    + profiles['total_fer'].rank(ascending=False)
    + profiles['income'].rank(ascending=True)
    + profiles['gdpp'].rank(ascending=True)
    + profiles['life_expec'].rank(ascending=True)
)
ordered_clusters = risk_score.sort_values(ascending=True).index.tolist()
segment_map = {
    ordered_clusters[0]: 'High Priority',
    ordered_clusters[1]: 'Developing',
    ordered_clusters[2]: 'Stable / Developed',
}

df_model['segment'] = df_model['kmeans_cluster'].map(segment_map)
df_model['priority_rank'] = df_model['segment'].map({'High Priority': 1, 'Developing': 2, 'Stable / Developed': 3})

segment_profiles = df_model.groupby('segment')[features + ['priority_rank']].mean().sort_values('priority_rank')
df_model.sort_values(['priority_rank', 'child_mort', 'gdpp'], ascending=[True, False, True]).to_csv(OUTPUT_DIR / 'country_segments.csv', index=False)
segment_profiles.to_csv(OUTPUT_DIR / 'segment_profiles.csv')

display(segment_profiles.round(2))
display(df_model[['country', 'segment', 'child_mort', 'income', 'life_expec', 'total_fer', 'gdpp']].sort_values(['priority_rank', 'child_mort'], ascending=[True, False]).head(15))

,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp,priority_rank
segment,,,,,,,,,,
High Priority,92.96,29.15,6.39,42.32,3942.40,12.02,59.19,5.01,1922.38,1.0
Developing,21.93,40.24,6.20,47.47,12305.60,7.60,72.81,2.31,6486.45,2.0
Stable / Developed,5.00,58.74,8.81,51.49,45672.22,2.67,80.13,1.75,42494.44,3.0


KeyError: 'priority_rank'

## PCA Visualization of Segments

In [8]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(x_scaled)
pca_df = pd.DataFrame({'pc1': coords[:, 0], 'pc2': coords[:, 1], 'country': df_model['country'], 'segment': df_model['segment']})
pca_df.to_csv(OUTPUT_DIR / 'pca_coordinates.csv', index=False)

plt.figure(figsize=(9, 6))
sns.scatterplot(data=pca_df, x='pc1', y='pc2', hue='segment', s=80)
plt.title('Country Segments Visualized with PCA')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'segment_pca_scatter.png', dpi=180)
plt.show()

## DBSCAN Clustering

DBSCAN is tested as a density-based alternative. Noise points represent unusual countries that do not fit dense groups well.

In [9]:
neighbors = NearestNeighbors(n_neighbors=5)
distances, _ = neighbors.fit(x_scaled).kneighbors(x_scaled)
kth_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 4))
plt.plot(kth_distances)
plt.title('DBSCAN 5-Nearest Neighbor Distance')
plt.xlabel('Sorted countries')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dbscan_k_distance.png', dpi=180)
plt.show()

dbscan_rows = []
for eps in np.arange(1.1, 3.1, 0.2):
    for min_samples in [3, 4, 5, 6, 8]:
        labels = DBSCAN(eps=float(eps), min_samples=min_samples).fit_predict(x_scaled)
        non_noise = labels != -1
        clusters = len(set(labels)) - (1 if -1 in labels else 0)
        if clusters >= 2 and non_noise.sum() > clusters:
            score = silhouette_score(x_scaled[non_noise], labels[non_noise])
        else:
            score = np.nan
        dbscan_rows.append({
            'eps': round(float(eps), 2),
            'min_samples': min_samples,
            'clusters': clusters,
            'noise_points': int((labels == -1).sum()),
            'silhouette_score': score,
            'ari_vs_kmeans': adjusted_rand_score(df_model['kmeans_cluster'], labels),
        })

dbscan_results = pd.DataFrame(dbscan_rows)
dbscan_results.to_csv(OUTPUT_DIR / 'dbscan_tuning.csv', index=False)
display(dbscan_results.dropna(subset=['silhouette_score']).sort_values(['silhouette_score', 'ari_vs_kmeans'], ascending=False).head(10))

,eps,min_samples,clusters,noise_points,silhouette_score,ari_vs_kmeans
4,1.1,8,2,97,0.440801,0.241725
8,1.3,6,2,48,0.413124,0.360239
7,1.3,5,2,46,0.407751,0.375552
1,1.1,4,3,64,0.402995,0.378062
9,1.3,8,3,53,0.369161,0.463435
0,1.1,3,4,58,0.346038,0.414223
3,1.1,6,4,75,0.303935,0.307950
2,1.1,5,4,71,0.297935,0.334442
10,1.5,3,2,25,0.226141,0.093427
15,1.7,3,2,20,0.213791,0.077561


## Classification Models

The final K-Means segment becomes the target label. This turns the unsupervised segmentation into a deployable predictive model that can classify new countries/markets.

In [10]:
x = df_model[features]
encoder = LabelEncoder()
y = encoder.fit_transform(df_model['segment'])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42, stratify=y)

try:
    from xgboost import XGBClassifier
    boosting_name = 'XGBoost'
    boosting_model = XGBClassifier(
        n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        objective='multi:softprob', eval_metric='mlogloss', random_state=42
    )
except Exception:
    boosting_name = 'Gradient Boosting (XGBoost fallback)'
    boosting_model = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=3000),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=2, random_state=42, class_weight='balanced'),
    boosting_name: boosting_model,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = []
reports = []
best_name, best_pipeline, best_accuracy = None, None, -1

for name, model in models.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    cv_scores = cross_val_score(pipeline, x, y, cv=cv, scoring='accuracy')
    pipeline.fit(x_train, y_train)
    preds = pipeline.predict(x_test)
    accuracy = accuracy_score(y_test, preds)
    metrics.append({'model': name, 'test_accuracy': accuracy, 'cv_accuracy_mean': cv_scores.mean(), 'cv_accuracy_std': cv_scores.std()})
    reports.append('\n' + name + '\n' + classification_report(y_test, preds, target_names=encoder.classes_))

    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name.lower().replace(' ', '_').replace('/', '')}_confusion_matrix.png", dpi=180)
    plt.show()

    if accuracy > best_accuracy:
        best_name, best_pipeline, best_accuracy = name, pipeline, accuracy

model_comparison = pd.DataFrame(metrics).sort_values(['test_accuracy', 'cv_accuracy_mean'], ascending=False)
model_comparison.to_csv(OUTPUT_DIR / 'classification_model_comparison.csv', index=False)
(OUTPUT_DIR / 'classification_reports.txt').write_text('\n'.join(reports), encoding='utf-8')
joblib.dump({'model': best_pipeline, 'label_encoder': encoder, 'features': features}, OUTPUT_DIR / 'best_classifier.joblib')

display(model_comparison)
print('Best model:', best_name)

,model,test_accuracy,cv_accuracy_mean,cv_accuracy_std
0,Logistic Regression,1.000000,0.975936,0.022632
2,Gradient Boosting (XGBoost fallback),0.976190,0.952228,0.035840
1,Random Forest,0.952381,0.946168,0.022502


Best model: Logistic Regression


## Feature Importance

In [12]:
if hasattr(best_pipeline.named_steps['model'], 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': features,
        'importance': best_pipeline.named_steps['model'].feature_importances_,
    }).sort_values('importance', ascending=False)
    importance_df.to_csv(OUTPUT_DIR / 'best_model_feature_importance.csv', index=False)

    plt.figure(figsize=(8, 5))
    sns.barplot(data=importance_df, x='importance', y='feature', color='#4C78A8')
    plt.title(f'Feature Importance: {best_name}')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'best_model_feature_importance.png', dpi=180)
    plt.show()
    display(importance_df)
else:
    print(f'{best_name} does not expose feature_importances_.')

Logistic Regression does not expose feature_importances_.


## Final Insights

- **High Priority** countries have high child mortality, lower GDP per capita, lower income, lower life expectancy, and higher fertility. These should be prioritized for intervention or resource allocation.
- **Developing** countries show mixed socio-economic conditions and can be targeted with growth, healthcare, and infrastructure programs.
- **Stable / Developed** countries show stronger income, GDP, and life expectancy, so they are lower urgency for aid-style interventions.
- K-Means gives the most interpretable segmentation for this dataset. DBSCAN is useful for identifying unusual/noise countries but is less directly business-friendly here.
- The supervised classifiers convert cluster insights into a repeatable prediction system for new records.

In [13]:
report = f"""# Customer Intelligence System - Country Segmentation

## Objective
Build an end-to-end intelligence pipeline using clustering and classification to identify priority country/customer segments from socio-economic indicators.

## Dataset
- Records: {len(df_model)} countries
- Features: {len(features)} numeric indicators
- Missing values: {int(df_model[features].isna().sum().sum())}

## Best Classification Model
{best_name} with test accuracy {best_accuracy:.3f}

## Segment Profiles
```
{segment_profiles.round(2).to_string()}
```

## Top High Priority Countries
```
{df_model[df_model['segment'] == 'High Priority'][['country', 'child_mort', 'income', 'life_expec', 'total_fer', 'gdpp']].sort_values(['child_mort', 'gdpp'], ascending=[False, True]).head(15).to_string(index=False)}
```

## Output Files
- country_segments.csv
- segment_profiles.csv
- classification_model_comparison.csv
- classification_reports.txt
- best_classifier.joblib
- EDA, clustering, PCA, DBSCAN, and confusion matrix charts
"""

(OUTPUT_DIR / 'assignment_report.md').write_text(report, encoding='utf-8')
print('Assignment complete. Outputs saved to:', OUTPUT_DIR)

Assignment complete. Outputs saved to: c:\Users\abhil\Desktop\Celebal\outputs\customer_intelligence
